<a href="https://colab.research.google.com/github/VarshaNeelvani/InnomaticsInternship/blob/main/Sentiment_Analysis_using_NLP_Pipeline_%26_ML_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ASSIGNMENT  NLP- 2 (Sentiment Analysis )**

## **Assignment: Sentiment Analysis using NLP Pipeline & ML Models**


In [1]:
!pip install pandas numpy scikit-learn nltk

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [11]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

### **Load Dataset**

In [3]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [5]:
df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print(df.info())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


In [7]:
print("Total samples:", len(df))

print("\nClass Distribution:")
print(df['review'].value_counts())

print("\nSample Texts:")
print(df['sentiment'].head())

Total samples: 50000

Class Distribution:
review
Loved today's show!!! It was a variety and not solely cooking (which would have been great too). Very stimulating and captivating, always keeping the viewer peeking around the corner to see what was coming up next. She is as down to earth and as personable as you get, like one of us which made the show all the more enjoyable. Special guests, who are friends as well made for a nice surprise too. Loved the 'first' theme and that the audience was invited to play along too. I must admit I was shocked to see her come in under her time limits on a few things, but she did it and by golly I'll be writing those recipes down. Saving time in the kitchen means more time with family. Those who haven't tuned in yet, find out what channel and the time, I assure you that you won't be disappointed.                                                                                                                                                               

In [8]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

In [12]:
df['clean_text'] = df['review'].apply(preprocess)

### **Train-Test Split**

In [14]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### **Feature Engineering**

In [15]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [16]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

### **Model Training**

In [17]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

LogisticRegression()

In [18]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

MultinomialNB()

In [19]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

DecisionTreeClassifier()

### **Model Evaluation**

In [20]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("-" * 50)

In [21]:
print("Logistic Regression")
evaluate(lr, X_test_tfidf, y_test)

print("Naive Bayes")
evaluate(nb, X_test_tfidf, y_test)

print("Decision Tree")
evaluate(dt, X_test_tfidf, y_test)

Logistic Regression
Accuracy: 0.8854
Precision: 0.885795791806026
Recall: 0.8854
F1 Score: 0.8853535547643645
--------------------------------------------------
Naive Bayes
Accuracy: 0.8468
Precision: 0.8468301299169001
Recall: 0.8468
F1 Score: 0.8467885018073846
--------------------------------------------------
Decision Tree
Accuracy: 0.7187
Precision: 0.7187856615958137
Recall: 0.7187
F1 Score: 0.7187002166010021
--------------------------------------------------


TF-IDF performed better than BoW because it reduces importance of common words.

Logistic Regression gave highest accuracy due to better generalization.

Naive Bayes was fastest but slightly less accurate.

Decision Tree overfitted the data.